# Lecture 7 — Class Exercise
## Part-to-Whole: Hierarchical Visualization

> **Push to:** `week07/lecture07_exercise.ipynb`


In [1]:
import pandas as pd
import plotly.express as px
import numpy as np

# Dataset: Global Energy Mix by Country and Source
df = pd.read_csv('../data/global_energy_mix.csv')

# Source type mapping — reuse from lecture
source_category = {
    'Coal': 'Fossil', 'Oil': 'Fossil', 'Natural Gas': 'Fossil',
    'Nuclear': 'Low-carbon', 'Hydro': 'Low-carbon',
    'Wind': 'Renewable', 'Solar': 'Renewable', 'Other Renewables': 'Renewable'
}
df['Source_Type'] = df['Source'].map(source_category)

print(f"Loaded: {len(df)} rows")
print(df.head(10))


Loaded: 103 rows
         Country         Region            Source  Share_pct     TWh  \
0  United States  North America              Coal         10  1015.0   
1  United States  North America               Oil         35  3220.0   
2  United States  North America       Natural Gas         34  3083.0   
3  United States  North America           Nuclear          9   798.0   
4  United States  North America             Hydro          3   339.0   
5  United States  North America              Wind          4   413.0   
6  United States  North America             Solar          3   325.0   
7  United States  North America  Other Renewables          2   229.0   
8          China           Asia              Coal         58  6929.1   
9          China           Asia               Oil         18  1620.0   

  Source_Type  
0      Fossil  
1      Fossil  
2      Fossil  
3  Low-carbon  
4  Low-carbon  
5   Renewable  
6   Renewable  
7   Renewable  
8      Fossil  
9      Fossil  


## Task 1 — Treemap: fossil fuel dependency by country

**What to build:** A treemap showing **fossil fuel TWh only**, broken down by Region → Country → Source (Coal / Oil / Natural Gas).

**Requirements:**
- Filter to fossil sources only before plotting
- Use `path=['Region', 'Country', 'Source']` for the hierarchy
- Colour encodes the fossil source type (Coal / Oil / Natural Gas) with a CVD-safe palette
- Show TWh values in labels — no percentages
- Grey out parent nodes (Region and Country level)
- Insight title naming which region or country is most fossil-dependent

> 💡 `df.loc[df['Source_Type'] == 'Fossil']`


In [2]:
# Task 1
df_fossil = df.loc[df['Source_Type'].eq('Fossil')].copy()

# Colorblind-safe palette; keep Coal's color for Task 3.
source_colors = {
    'Coal': '#0072B2',
    'Oil': '#E69F00',
    'Natural Gas': '#009E73',
}

fossil_share = df_fossil.groupby('Country')['Share_pct'].sum()
leader = fossil_share.idxmax()

fig1 = px.treemap(
    df_fossil,
    path=['Region', 'Country', 'Source'],
    values='TWh',
    color='Source',
    color_discrete_map=source_colors,
    title=f'{leader} has the highest fossil-fuel share ({fossil_share.max():.0f}%)',
    template='plotly_white',
)
fig1.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
    root_color='#DDDDDD',
)
fig1.data[0].marker.colors = [
    color if color in source_colors.values() else '#DDDDDD'
    for color in fig1.data[0].marker.colors
]
fig1.show()


## Task 2 — Treemap vs bar: low-carbon energy by country

**What to build:** Build **both** a treemap and a horizontal bar chart showing total low-carbon TWh (Nuclear + Hydro) per country. Then answer the question in a markdown cell below.

**Requirements:**
- Filter to `Source_Type == 'Low-carbon'` and aggregate TWh by country
- Treemap: single-level `path=['All', 'Country']` with a dummy root node labelled `'Low-carbon'`
- Bar chart: sorted by TWh, horizontal orientation, CVD-safe colour
- Both charts show TWh values, not percentages
- Insight title on the bar chart naming the leading country

In [3]:
# Task 2 — charts
df_low = (
    df.loc[df['Source_Type'].eq('Low-carbon')]
    .groupby('Country', as_index=False)['TWh'].sum()
)
df_low['All'] = 'Low-carbon'

fig2_treemap = px.treemap(
    df_low,
    path=['All', 'Country'],
    values='TWh',
    color='All',
    color_discrete_map={'Low-carbon': '#2E75B6'},
    title='Low-carbon energy by country',
    template='plotly_white',
)
fig2_treemap.update_traces(
    texttemplate='%{label}<br>%{value:,.0f} TWh',
    hovertemplate='<b>%{label}</b><br>%{value:,.0f} TWh<extra></extra>',
)
fig2_treemap.show()

leader = df_low.loc[df_low['TWh'].idxmax(), 'Country']

fig2_bar = px.bar(
    df_low,
    x='TWh',
    y='Country',
    orientation='h',
    color_discrete_sequence=['#2E75B6'],
    title=f'{leader} leads in low-carbon energy',
    labels={'TWh': 'Energy (TWh)'},
    template='plotly_white',
)
fig2_bar.update_layout(yaxis={'categoryorder': 'total ascending'})
fig2_bar.update_traces(
    texttemplate='%{x:,.0f} TWh',
    textposition='outside',
    cliponaxis=False,
)
fig2_bar.show()

### Task 2 — reflection

**Which chart tells the ranking story more clearly, and why?**

*Ans : I think the horizontal bar chart makes the ranking easier to understand because I can compare the countries’ totals directly. The treemap is useful for showing each country’s share of the whole, but it’s harder to see the exact order. France has the highest low-carbon total.*

## Task 3 — 100% stacked bar: which region relies most on coal?

**What to build:** A horizontal 100% stacked bar chart showing, for each **Region**, how its fossil energy splits between Coal, Oil and Natural Gas. This is the **same data as your Task 1 treemap**, visualised a second way.

**Requirements:**
- Reuse `df_fossil` from Task 1; aggregate TWh by `Region` × `Source` and convert to **shares of each region's fossil total**
- Horizontal bars, regions **sorted by coal share** (highest at the top)
- Coal is the segment on the baseline (first in `category_orders`) and the **only coloured segment** — keep it the same colour as in Task 1; Oil and Natural Gas in two greys
- Percentage labels on the coal segments only
- Insight title naming the most coal-dependent region and its coal share

> 💡 `df.groupby('Region')['TWh'].transform('sum')` gives each row its region total

In [4]:
# Task 3
mix = df_fossil.groupby(['Region', 'Source'], as_index=False)['TWh'].sum()
mix['Share'] = mix['TWh'] / mix.groupby('Region')['TWh'].transform('sum')

coal_share = mix.loc[mix['Source'].eq('Coal')].set_index('Region')['Share']
region_order = coal_share.sort_values().index.tolist()
leader = coal_share.idxmax()

mix['Label'] = mix['Share'].map(lambda value: f'{value:.0%}').where(
    mix['Source'].eq('Coal'), ''
)

fig3 = px.bar(
    mix,
    x='Share',
    y='Region',
    color='Source',
    text='Label',
    orientation='h',
    barmode='stack',
    color_discrete_map={
        'Coal': '#0072B2',
        'Oil': '#B0B0B0',
        'Natural Gas': '#D9D9D9',
    },
    category_orders={
        'Source': ['Coal', 'Oil', 'Natural Gas'],
        'Region': region_order,
    },
    title=f'{leader} relies most on coal ({coal_share[leader]:.0%} of fossil energy)',
    labels={'Share': 'Share of regional fossil energy', 'Source': 'Fossil source'},
    template='plotly_white',
)
fig3.update_traces(textposition='inside', insidetextanchor='middle')
fig3.update_layout(
    xaxis={'tickformat': '.0%', 'range': [0, 1]},
    yaxis={'title': ''},
    legend={'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'x': 0},
)
fig3.show()

### Task 3 — reflection

**Your Task 1 treemap and this stacked bar show the same fossil data. Name one question each chart answers that the other cannot.**

*Ans : The treemap answers, “Which countries use the most fossil energy, and how much comes from coal, oil, and natural gas?” The stacked bar answers, “Which region depends most on coal within its fossil-energy mix?” Africa has the highest coal share, at about 78%.*